# Vòng 60c: nén gallery ở quy mô ImageNet-1k

Gallery 1,28 triệu x 2048 (21 mảnh fp16, từ notebook `cure-in1k-extract-full`), phía truy vấn từ Dataset `cure-in1k-query`.
Các cấu hình giữ đủ 1,28 triệu vector (g0, bf16, ivf, hnsw) chạy **một tiến trình** vì gallery chiếm 5-10 GB;
các cấu hình thu nhỏ gallery chạy 4 tiến trình. Kết quả nén vào `/kaggle/working/edge_gallery_in1k_results.zip`.

In [ ]:
TASKS = ["latency"]
BIG = []
SMALL_WORKERS = 1


In [ ]:
print("@@CELL 1@@", flush=True)
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
T0 = time.time()
def tick(m): print(f"[{(time.time() - T0) / 60:6.1f} phút] {m}", flush=True)

code_dir = next(t.parent for t in sorted(INPUT.rglob("pyproject.toml")) if 'name = "cure-ood"' in t.read_text(encoding="utf-8"))
REPO = WORK / "CURE"
if not REPO.exists():
    shutil.copytree(code_dir, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))  # cài dạng -e ko có hiệu lực ngay trong tiến trình đang chạy
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"], capture_output=True, text=True)
print("faiss:", "ok" if r.returncode == 0 else "LỖI: " + r.stderr[-300:])

# Gom đặc trưng và mảnh gallery vào một thư mục phẳng (symlink, ko tốn chỗ).
FEAT = WORK / "features"; FEAT.mkdir(exist_ok=True)
for f in INPUT.rglob("*.np[yz]"):
    if f.suffix == ".npz" or f.name.startswith(("feats_", "labels_")):
        if not (FEAT / f.name).exists():
            (FEAT / f.name).symlink_to(f)
os.environ.update(CURE_FEATURES=str(FEAT), CURE_HF_AUTO="0")
os.chdir(REPO)
shards = sorted(FEAT.glob("feats_*.npy"))
print(open("VERSION.txt").read())
print(len(shards), "mảnh gallery,", sorted(p.name for p in FEAT.glob("*.npz")))

In [ ]:
print("@@CELL 2@@", flush=True)
import numpy as np
from cure_ood.data import Benchmark
from cure_ood.eval.runner import Experiment
EXP = "edge_gallery_in1k"
exp = Experiment.load(EXP)
b = Benchmark.load("imagenet1k")
sh = b.shards
print("gallery:", sh.size, "x", sh.dim, "| nhãn:", len(sh.labels), "lớp", len(np.unique(sh.labels)))
assert sh.size >= 1_281_167, f"thiếu ảnh train: {sh.size}"
print("C:", b.C.shape, "| luồng ID:", b.test_feats.shape, "| val OOD:", b.val_ood_data()[0].shape)
print("tập OOD:", {d: len(b.array(f"{d}_labels")) for d in b.domains})
tick("kiểm dữ liệu xong")

In [ ]:
print("@@CELL 4@@", flush=True)
import zipfile
print(subprocess.run(["lscpu"], capture_output=True, text=True).stdout[:400])
env = {**os.environ, "CUDA_VISIBLE_DEVICES": "-1"}
# Gallery 1,28 triệu: mỗi truy vấn ~1 giây trên 4 nhân, nên ít bước hơn vòng 60 (p50 vẫn ổn định).
subprocess.run([sys.executable, "scripts/edge_latency.py", "--experiment", "edge_gallery_in1k",
                "--bases", "imagenet1k", "--threads", "4", "--warm", "10", "--steps", "100", "--reps", "3"],
               env=env, check=False)
out = Path("/kaggle/working/in1k_latency.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in (REPO / "results").rglob("*.json"):
        z.write(f, f.relative_to(REPO))
print(out, round(out.stat().st_size / 1e3, 1), "KB")